In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

/kaggle/input/english-to-hindi-parallel-dataset/newdata.csv


In [2]:
import numpy as np
import pandas as pd

import keras
from keras.models import Model
from keras.layers import Input, LSTM, Dense,TimeDistributed,Embedding,Bidirectional
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from string import digits
import nltk
import re
import string
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', -1)

Using TensorFlow backend.


In [3]:
lines = pd.read_csv('/kaggle/input/english-to-hindi-parallel-dataset/newdata.csv')
lines = lines[:30000]
lines.head()

,Unnamed: 0,english_sentence,hindi_sentence
0,0,politicians do not have permission to do what needs to be done.,"राजनीतिज्ञों के पास जो कार्य करना चाहिए, वह करने कि अनुमति नहीं है ."
1,1,"I'd like to tell you about one such child,","मई आपको ऐसे ही एक बच्चे के बारे में बताना चाहूंगी,"
2,2,This percentage is even greater than the percentage in India.,यह प्रतिशत भारत में हिन्दुओं प्रतिशत से अधिक है।
3,3,what we really mean is that they're bad at not paying attention.,हम ये नहीं कहना चाहते कि वो ध्यान नहीं दे पाते
4,4,.The ending portion of these Vedas is called Upanishad.,इन्हीं वेदों का अंतिम भाग उपनिषद कहलाता है।


In [4]:
# Lowercase all characters
lines['english_sentence']=lines['english_sentence'].apply(lambda x: str(x))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: str(x))
lines['english_sentence']=lines['english_sentence'].apply(lambda x: x.lower())
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: x.lower())

In [5]:
# Remove quotes
lines['english_sentence']=lines['english_sentence'].apply(lambda x: re.sub("'", '', x))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: re.sub("'", '', x))


In [6]:
exclude = set(string.punctuation) # Set of all special characters
# Remove all the special characters
lines['english_sentence']=lines['english_sentence'].apply(lambda x: ''.join(ch for ch in x if ch not in exclude))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: ''.join(ch for ch in x if ch not in exclude))

In [7]:
# Remove all numbers from text
remove_digits = str.maketrans('', '', digits)
lines['english_sentence']=lines['english_sentence'].apply(lambda x: x.translate(remove_digits))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: x.translate(remove_digits))

lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x: re.sub("[२३०८१५७९४६]", "", x))

# Remove extra spaces
lines['english_sentence']=lines['english_sentence'].apply(lambda x: x.strip())
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: x.strip())
lines['english_sentence']=lines['english_sentence'].apply(lambda x: re.sub(" +", " ", x))
lines['hindi_sentence']=lines['hindi_sentence'].apply(lambda x: re.sub(" +", " ", x))


In [8]:
# Add start and end tokens to target sequences
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x : 'START_ '+ x + ' _END')

In [9]:
### Get English and Hindi Vocabulary
all_eng_words=set()
for eng in lines['english_sentence']:
    for word in eng.split():
        if word not in all_eng_words:
            all_eng_words.add(word)

all_hindi_words=set()
for hin in lines['hindi_sentence']:
    for word in hin.split():
        if word not in all_hindi_words:
            all_hindi_words.add(word)

In [10]:
lines['length_eng_sentence']=lines['english_sentence'].apply(lambda x:len(x.split(" ")))
lines['length_hin_sentence']=lines['hindi_sentence'].apply(lambda x:len(x.split(" ")))

In [11]:
lines.head()
lines[lines['length_eng_sentence']>30].shape

(2919, 5)

In [12]:
lines=lines[lines['length_eng_sentence']<=20]
lines=lines[lines['length_hin_sentence']<=20]

In [13]:
print("maximum length of Hindi Sentence ",max(lines['length_hin_sentence']))
print("maximum length of English Sentence ",max(lines['length_eng_sentence']))

maximum length of Hindi Sentence  20
maximum length of English Sentence  20


In [14]:
max_length_src=max(lines['length_hin_sentence'])
max_length_tar=max(lines['length_eng_sentence'])

In [15]:
input_words = sorted(list(all_eng_words))
target_words = sorted(list(all_hindi_words))
num_encoder_tokens = len(all_eng_words)
num_decoder_tokens = len(all_hindi_words)
num_encoder_tokens, num_decoder_tokens

(33708, 40802)

In [16]:
num_decoder_tokens += 1

In [17]:
#create dictionaries that map each unique word in the input and target vocabularies to a unique integer index, 
#which will be used to represent the words as one-hot encoded vectors during training and inference.

#creates a dictionary input_token_index where each unique word in input_words (a list of all words in the input vocabulary) 
#is mapped to a unique integer index. The index starts at 1 (not 0) because 0 will be used as a padding token. 
input_token_index = dict([(word, i+1) for i, word in enumerate(input_words)])

#The enumerate() function generates a sequence of (index, word) pairs for each word in input_words/target words, which are then converted to a dictionary 
#using the dict() function.
target_token_index = dict([(word, i+1) for i, word in enumerate(target_words)])

In [18]:
#creates a dictionary reverse_input_char_index where each integer index in input_token_index is mapped to the corresponding word. 
reverse_input_char_index = dict((i, word) for word, i in input_token_index.items())

#The items() method of the input_token_index/target_token_index dictionary returns a sequence of (word, index) pairs, which are then reversed 
#and converted to a dictionary using the dict() function.
reverse_target_char_index = dict((i, word) for word, i in target_token_index.items())

In [19]:
lines.head(10)

,Unnamed: 0,english_sentence,hindi_sentence,length_eng_sentence,length_hin_sentence
0,0,politicians do not have permission to do what needs to be done,START_ राजनीतिज्ञों के पास जो कार्य करना चाहिए वह करने कि अनुमति नहीं है _END,12,15
1,1,id like to tell you about one such child,START_ मई आपको ऐसे ही एक बच्चे के बारे में बताना चाहूंगी _END,9,13
2,2,this percentage is even greater than the percentage in india,START_ यह प्रतिशत भारत में हिन्दुओं प्रतिशत से अधिक है। _END,10,11
3,3,what we really mean is that theyre bad at not paying attention,START_ हम ये नहीं कहना चाहते कि वो ध्यान नहीं दे पाते _END,12,13
4,4,the ending portion of these vedas is called upanishad,START_ इन्हीं वेदों का अंतिम भाग उपनिषद कहलाता है। _END,9,10
6,6,in this lies the circumstances of people before you,START_ इसमें तुमसे पूर्व गुज़रे हुए लोगों के हालात हैं। _END,9,11
7,7,and who are we to say even that they are wrong,START_ और हम होते कौन हैं यह कहने भी वाले कि वे गलत हैं _END,11,15
10,10,please ensure that you use the appropriate form,START_ कृपया यह सुनिश्चित कर लें कि आप सही फॉर्म का प्रयोग कर रहें हैं _END,8,16
11,11,category religious text,START_ श्रेणीधर्मग्रन्थ _END,3,3
12,12,this period summarily is pepped up with devotion,START_ यह काल समग्रतः भक्ति भावना से ओतप्रोत काल है। _END,8,11


In [20]:
from sklearn.model_selection import train_test_split
X, y = lines['english_sentence'], lines['hindi_sentence']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,random_state=42)
X_train.shape, X_test.shape

((16007,), (4002,))

In [21]:
X_train

21471    the future development of the mini steel industry will depend on the availability of sponge iron
2807     and god“ he said to the poor man ”is answering your plea                                        
1577     deep shade of white mausoleum could clearly be seen in the lake                                 
7990     hindu philosophy of unity                                                                       
20023    to stand up and join in                                                                         
                  ...                                                                                    
16973    hindu people really do not worship stones and iron as some people explain                       
17972    he is self originated and maker of world                                                        
8002     during this period considerable political consciousness developed in the country                
1303     badruddin himself had to learn all th

In [22]:
def generate_batch(X = X_train, y = y_train, batch_size = 128):
    ''' Generate a batch of data '''
    while True:
        for j in range(0, len(X), batch_size):
            encoder_input_data = np.zeros((batch_size, max_length_src),dtype='float32')
            decoder_input_data = np.zeros((batch_size, max_length_tar),dtype='float32')
            decoder_target_data = np.zeros((batch_size, max_length_tar, num_decoder_tokens),dtype='float32')
            for i, (input_text, target_text) in enumerate(zip(X[j:j+batch_size], y[j:j+batch_size])):
                for t, word in enumerate(input_text.split()):
                    encoder_input_data[i, t] = input_token_index[word] # encoder input seq
                for t, word in enumerate(target_text.split()):
                    if t<len(target_text.split())-1:
                        decoder_input_data[i, t] = target_token_index[word] # decoder input seq
                    if t>0:
                        # decoder target sequence (one hot encoded)
                        # does not include the START_ token
                        # Offset by one timestep
                        decoder_target_data[i, t - 1, target_token_index[word]] = 1.
            yield([encoder_input_data, decoder_input_data], decoder_target_data)

In [23]:
#this code sets up an LSTM-based encoder for a sequence-to-sequence model, which takes an input sequence of variable length, 
#processes it with an embedding layer and an LSTM layer, and outputs the final hidden and cell states of the LSTM.

latent_dim = 300   #sets the dimensionality of the hidden state of the LSTM layer.
# Encoder
#is a placeholder for the input sequence, with shape (None,). This allows the model to accept input sequences of variable length.
encoder_inputs = Input(shape=(None,)) 

#The num_encoder_tokens parameter specifies the number of unique tokens in the input vocabulary, 
#and mask_zero=True indicates that any padding tokens in the input sequence should be masked out.
enc_emb =  Embedding(num_encoder_tokens, latent_dim, mask_zero = True)(encoder_inputs)

#creates an LSTM layer with latent_dim units, and return_state=True instructs the 
#layer to return the final hidden state and cell state of the LSTM after processing the input sequence.
encoder_lstm = LSTM(latent_dim, return_state=True)

encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

# We discard `encoder_outputs` and only keep the states.
# creates a list containing the final hidden state and cell state of the encoder LSTM, 
# which will be used as the initial state for the decoder LSTM in the sequence-to-sequence model.
encoder_states = [state_h, state_c]

In [24]:
#creates a dense layer with a softmax activation function, which maps the output of the decoder LSTM to a 
#probability distribution over the vocabulary of the output language.

decoder_inputs = Input(shape=(None,)) #is a placeholder for the decoder input sequence, with shape (None,).

#creates an embedding layer for the decoder input sequence, similar to the one used for the encoder input sequence.
dec_emb_layer = Embedding(num_decoder_tokens, latent_dim, mask_zero = True)

#applies the embedding layer to the decoder input sequence.
dec_emb = dec_emb_layer(decoder_inputs)

# We set up our decoder to return full output sequences, and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.

#creates an LSTM layer for the decoder with latent_dim units, which returns the full output sequence 
#(return_sequences=True) as well as the final hidden and cell states (return_state=True).
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

#applies the decoder LSTM to the embedded decoder input sequence (dec_emb) with the initial hidden and 
#cell states set to the final states of the encoder LSTM (encoder_states).
#Note that the second and third return values of the LSTM (the cell state and the final state) are ignored with _.
decoder_outputs, _, _ = decoder_lstm(dec_emb,
                                     initial_state=encoder_states)

#creates a dense layer with a softmax activation function, which maps the output of the decoder 
#LSTM to a probability distribution over the vocabulary of the output language.
decoder_dense = Dense(num_decoder_tokens, activation='softmax')

decoder_outputs = decoder_dense(decoder_outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [25]:
#optimizer='adam': sets the Adam optimization algorithm as the optimizer to be used during training. 
#Adam is a popular optimization algorithm that dynamically adjusts the learning rate during training to improve 
#convergence speed and stability.

#loss='categorical_crossentropy': sets the categorical cross-entropy loss function as the objective to be minimized during training. 
#Categorical cross-entropy is a common loss function for multi-class classification problems, and is well-suited for training models 
#that output probability distributions over a set of classes.


#metrics=['accuracy']: sets the accuracy metric to be used during training. Accuracy is a common metric for classification problems, 
#which measures the proportion of correctly predicted labels among all predictions.

model.compile(optimizer='adam', loss='categorical_crossentropy',metrics=['accuracy'])

In [26]:
model.summary()
train_samples = len(X_train)
val_samples = len(X_test)
batch_size = 64
epochs = 100

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            (None, None)         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            (None, None)         0                                            
__________________________________________________________________________________________________
embedding_1 (Embedding)         (None, None, 300)    10112400    input_1[0][0]                    
__________________________________________________________________________________________________
embedding_2 (Embedding)         (None, None, 300)    12240900    input_2[0][0]                    
____________________________________________________________________________________________

In [27]:
model.fit_generator(generator = generate_batch(X_train, y_train, batch_size = batch_size),
                    steps_per_epoch = train_samples//batch_size,
                    epochs=5,
                    validation_data = generate_batch(X_test, y_test, batch_size = batch_size),
                    validation_steps = val_samples//batch_size)



/opt/conda/lib/python3.6/site-packages/tensorflow_core/python/framework/indexed_slices.py:433: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "


Epoch 1/5
250/250 [==============================] - 168s 673ms/step - loss: 3.7389 - accuracy: 0.1133 - val_loss: 3.6945 - val_accuracy: 0.1199
Epoch 2/5
250/250 [==============================] - 165s 659ms/step - loss: 3.2724 - accuracy: 0.1368 - val_loss: 3.0422 - val_accuracy: 0.1436
Epoch 3/5
250/250 [==============================] - 165s 660ms/step - loss: 3.0760 - accuracy: 0.1588 - val_loss: 3.4962 - val_accuracy: 0.1607
Epoch 4/5
250/250 [==============================] - 165s 659ms/step - loss: 2.9010 - accuracy: 0.1814 - val_loss: 3.1662 - val_accuracy: 0.1733
Epoch 5/5
250/250 [==============================] - 165s 660ms/step - loss: 2.7468 - accuracy: 0.1997 - val_loss: 3.1484 - val_accuracy: 0.1823


In [40]:
#saves the trained model to a file called 'eng-to-hindi.h5' using the HDF5 file format.
#The HDF5 format is a data model, library, and file format for storing and managing large amounts of numerical data.
#By saving the trained model to a file, you can later load the model back into memory and use it for prediction 
#without having to retrain it from scratch.

model.save('eng-to-hindi.h5')

In [41]:
train_gen = generate_batch(X_train, y_train, batch_size = 1)
k=-1

In [42]:
#This code sets up the inference model for the seq2seq model. The inference model is used to generate predictions for 
#new input sequences that were not seen during training.

#The encoder model takes the input sequence and generates the hidden state and cell state vectors, which represent the
#"thought vectors" for the input sequence. These vectors are then used to initialize the decoder model.
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder setup
# Below tensors will hold the states of the previous time step
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2= dec_emb_layer(decoder_inputs) # Get the embeddings of the decoder sequence

# To predict the next word in the sequence, set the initial states to the states from the previous time step
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2) # A dense softmax layer to generate prob dist. over the target vocabulary

# Final decoder model
#The decoder model takes the "thought vectors" and the decoder input sequence and generates the output sequence. 
#In this case, the decoder input sequence is just a start-of-sequence token followed by an empty sequence. At each time step, 
#the model generates a probability distribution over the target vocabulary using a dense softmax layer. The word with the highest 
#probability is chosen as the output at each time step.
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2)

In [43]:
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)  # Encode the input as state vectors.
    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1,1))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0] = target_token_index['START_']

    # Sampling loop for a batch of sequences (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])  # Sample a token
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += ' '+sampled_char

        # Exit condition: either hit max length or find stop character.
        if (sampled_char == '_END' or
           len(decoded_sentence) > 50):
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1,1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]
    return decoded_sentence

In [47]:
k+=1
(input_seq, actual_output), _ = next(train_gen)
decoded_sentence = decode_sequence(input_seq)
print('Input English sentence:', X_train[k:k+1].values[0])
print('Actual Hindi Translation:', y_train[k:k+1].values[0][6:-4])
print('Predicted Hindi Translation:', decoded_sentence[:-4])

Input English sentence: hindu philosophy of unity
Actual Hindi Translation:  हिन्दुत्व एकत्व का दर्शन है 
Predicted Hindi Translation:  भारत में एक प्रकार 


In [33]:
k+=1
(input_seq, actual_output), _ = next(train_gen)
decoded_sentence = decode_sequence(input_seq)
print('Input English sentence:', X_train[k:k+1].values[0])
print('Actual Hindi Translation:', y_train[k:k+1].values[0][6:-4])
print('Predicted Hindi Translation:', decoded_sentence[:-4])

Input English sentence: and god“ he said to the poor man ”is answering your plea
Actual Hindi Translation:  “और भगवान्” उन्होंने गरीब व्यक्ति से कहा “तुम्हारी प्रार्थना सुन रहे हैं 
Predicted Hindi Translation:  और यह एक बार एक तरह से एक तरह से नहीं है 


In [34]:
k+=1
(input_seq, actual_output), _ = next(train_gen)
decoded_sentence = decode_sequence(input_seq)
print('Input English sentence:', X_train[k:k+1].values[0])
print('Actual Hindi Translation:', y_train[k:k+1].values[0][6:-4])
print('Predicted Hindi Translation:', decoded_sentence[:-4])

Input English sentence: deep shade of white mausoleum could clearly be seen in the lake
Actual Hindi Translation:  श्वेत मकबरे की गहरी छाया को स्पष्ट देखा जा सकता था उस सरोवर में। 
Predicted Hindi Translation:  इस प्रकार में एक प्रकार में एक बार से भी है। 
